In [ ]:
# x-y-v_x phase-space animation helpers (parallel frame render)

import glob
import os
import re
import shutil
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import List, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import numpy as np
import pandas as pd
from IPython.display import Video, display
from tqdm import tqdm

from xyvx_render_worker import render_one_frame

PATTERN = "PIC_Part_*.csv"
OUT_MP4 = "xyvx_animation.mp4"
FPS = 10
DPI = 100
SCATTER_SIZE = 2
SCATTER_ALPHA = 0.7
DROP_FRAC = 0.0
SUBSAMPLE_SEED = 0
PHASE_DENSITY_BINS = 64
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
FRAME_CACHE_DIR = Path("_xyvx_frame_cache")
# Physical time step used when writing PIC_Part_XXXXXX.csv (t = step * DT).
DT = 0.1


def extract_step(path: Path) -> int:
    m = re.search(r"PIC_Part_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else -1


def physical_time(path: Path, dt: float = DT) -> float:
    """CSV is written after Step, so t = step * dt."""
    return extract_step(path) * dt


def particle_mass_and_domain(df: pd.DataFrame) -> Tuple[float, float, float]:
    """Auto-normalized mass m = Lx*Ly/N (matches electrostatic-pic)."""
    x = df["X"].to_numpy()
    y = df["Y"].to_numpy()
    lx = float(x.max() - x.min())
    ly = float(y.max() - y.min())
    n = max(len(x), 1)
    return (lx * ly) / n, lx, ly


def vx_from_frame(df: pd.DataFrame, mass: float | None = None) -> np.ndarray:
    """Convert stored momentum_0 to v_x using particle mass."""
    if mass is None:
        mass, _, _ = particle_mass_and_domain(df)
    return df["momentum_0"].to_numpy() / mass


def list_snapshot_files(pattern: str = PATTERN) -> Tuple[Path, List[Path]]:
    files = sorted(glob.glob(pattern), key=lambda f: extract_step(Path(f)))
    if not files:
        raise FileNotFoundError(f"No files matching pattern: {pattern}")
    if len(files) < 2:
        raise ValueError("Need at least two CSV files: init (step 1) plus later frames.")
    paths = [Path(f) for f in files]
    return paths[0], paths[1:]


def classify_beams(init_file: Path) -> Tuple[np.ndarray, np.ndarray]:
    required = {"id", "X", "Y", "momentum_0"}
    df_init = pd.read_csv(init_file, usecols=list(required))
    if not required.issubset(df_init.columns):
        raise ValueError(f"{init_file} missing required columns.")

    mass, _, _ = particle_mass_and_domain(df_init)
    vx_init = vx_from_frame(df_init, mass)
    left_ids = df_init.loc[vx_init < 0, "id"].to_numpy()
    right_ids = df_init.loc[vx_init >= 0, "id"].to_numpy()
    print(f"Init file: {init_file.name}")
    print(f"Left-moving ids (vx < 0): {len(left_ids)}")
    print(f"Right-moving ids (vx >= 0): {len(right_ids)}")
    print(f"Inferred particle mass m = Lx*Ly/N = {mass:.6g}")
    return left_ids, right_ids


def subsample_ids(left_ids: np.ndarray, right_ids: np.ndarray,
                  drop_frac: float = DROP_FRAC,
                  seed: int = SUBSAMPLE_SEED) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    all_ids = np.unique(np.concatenate([left_ids, right_ids]))
    n_keep = max(1, int(round((1.0 - drop_frac) * len(all_ids))))
    rng = np.random.default_rng(seed)
    keep_ids = rng.choice(all_ids, size=n_keep, replace=False)
    left_kept = left_ids[np.isin(left_ids, keep_ids)]
    right_kept = right_ids[np.isin(right_ids, keep_ids)]
    print(
        f"Subsample: keep {len(keep_ids)} / {len(all_ids)} particles "
        f"({100 * (1 - drop_frac):.0f}% kept, drop {100 * drop_frac:.0f}%)"
    )
    return keep_ids, left_kept, right_kept


def load_animation_frames(anim_files: List[Path]) -> List[pd.DataFrame]:
    required = {"id", "X", "Y", "momentum_0"}
    frames = []
    for f in tqdm(anim_files, desc="Reading CSV frames"):
        df = pd.read_csv(f, usecols=list(required))
        if not required.issubset(df.columns):
            raise ValueError(f"{f} missing columns. Found: {list(df.columns)}")
        frames.append(df)
    return frames


def _padded_lim(vals: np.ndarray) -> Tuple[float, float]:
    pad = (vals.max() - vals.min()) * 0.02 if np.ptp(vals) != 0 else 1.0
    return float(vals.min() - pad), float(vals.max() + pad)


def compute_axis_limits(
    frames: List[pd.DataFrame],
) -> Tuple[Tuple[float, float], Tuple[float, float], Tuple[float, float]]:
    mass, _, _ = particle_mass_and_domain(frames[0])
    all_x = np.concatenate([df["X"].to_numpy() for df in frames])
    all_y = np.concatenate([df["Y"].to_numpy() for df in frames])
    all_vx = np.concatenate([vx_from_frame(df, mass) for df in frames])
    return _padded_lim(all_x), _padded_lim(all_y), _padded_lim(all_vx)


def compute_phase_space_bin_density(
    x: np.ndarray,
    y: np.ndarray,
    vx: np.ndarray,
    x_edges: np.ndarray,
    y_edges: np.ndarray,
    vx_edges: np.ndarray,
    mass: float,
) -> np.ndarray:
    """Mass-weighted phase-space density: f = (n_bin * m) / (dx*dy*dvx)."""
    hist, _ = np.histogramdd((x, y, vx), bins=(x_edges, y_edges, vx_edges))
    dx = float(x_edges[1] - x_edges[0])
    dy = float(y_edges[1] - y_edges[0])
    dv = float(vx_edges[1] - vx_edges[0])
    bin_vol = max(dx * dy * dv, 1e-30)
    dens = hist * (mass / bin_vol)
    x_idx = np.clip(np.searchsorted(x_edges, x, side="right") - 1, 0, len(x_edges) - 2)
    y_idx = np.clip(np.searchsorted(y_edges, y, side="right") - 1, 0, len(y_edges) - 2)
    vx_idx = np.clip(np.searchsorted(vx_edges, vx, side="right") - 1, 0, len(vx_edges) - 2)
    return dens[x_idx, y_idx, vx_idx]


def prepare_phase_space_density(
    frames: List[pd.DataFrame], keep_ids: np.ndarray, num_bins: int = PHASE_DENSITY_BINS
) -> Tuple[List[dict], Tuple[float, float]]:
    """Return compact numpy frame dicts colored by phase-space density f."""
    mass, lx, ly = particle_mass_and_domain(frames[0])
    print(f"Density uses m = {mass:.6g}, Lx ≈ {lx:.6g}, Ly ≈ {ly:.6g}")

    all_x = np.concatenate([df["X"].to_numpy() for df in frames])
    all_y = np.concatenate([df["Y"].to_numpy() for df in frames])
    all_vx = np.concatenate([vx_from_frame(df, mass) for df in frames])

    def _edges(vals: np.ndarray) -> np.ndarray:
        lo, hi = float(vals.min()), float(vals.max())
        if lo == hi:
            lo -= 1.0
            hi += 1.0
        return np.linspace(lo, hi, num_bins + 1)

    x_edges = _edges(all_x)
    y_edges = _edges(all_y)
    vx_edges = _edges(all_vx)

    keep_set = set(keep_ids.tolist())
    compact = []
    f_min = np.inf
    f_max = 0.0
    for df in tqdm(frames, desc="Binning phase-space density"):
        x = df["X"].to_numpy()
        y = df["Y"].to_numpy()
        vx = vx_from_frame(df, mass)
        f_vals = compute_phase_space_bin_density(
            x, y, vx, x_edges, y_edges, vx_edges, mass
        )
        keep_mask = df["id"].isin(keep_set).to_numpy()
        x_k = x[keep_mask].astype(np.float32, copy=False)
        y_k = y[keep_mask].astype(np.float32, copy=False)
        vx_k = vx[keep_mask].astype(np.float32, copy=False)
        f_k = f_vals[keep_mask].astype(np.float32, copy=False)
        if f_k.size:
            f_min = min(f_min, float(f_k.min()))
            f_max = max(f_max, float(f_k.max()))
        compact.append({"x": x_k, "y": y_k, "vx": vx_k, "rho": f_k})
    if not np.isfinite(f_min):
        f_min = 0.0
    return compact, (f_min, max(f_max, f_min + 1e-30))


def render_animation_parallel(
    frames: List[dict],
    anim_files: List[Path],
    xlim,
    ylim,
    vxlim,
    rho_lim,
    out_mp4: str = OUT_MP4,
    fps: int = FPS,
    n_workers: int = N_WORKERS,
    cache_dir: Path = FRAME_CACHE_DIR,
    dt: float = DT,
) -> str:
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
    png_dir = cache_dir / "png"
    npz_dir = cache_dir / "npz"
    png_dir.mkdir(parents=True)
    npz_dir.mkdir(parents=True)

    npz_paths = []
    png_paths = []
    for i, fr in enumerate(tqdm(frames, desc="Caching frame arrays")):
        npz_path = npz_dir / f"frame_{i:06d}.npz"
        png_path = png_dir / f"frame_{i:06d}.png"
        np.savez_compressed(npz_path, x=fr["x"], y=fr["y"], vx=fr["vx"], rho=fr["rho"])
        npz_paths.append(npz_path)
        png_paths.append(png_path)

    n_frames = len(frames)
    tasks = [
        (
            i,
            str(npz_paths[i]),
            str(png_paths[i]),
            xlim,
            ylim,
            vxlim,
            rho_lim,
            physical_time(anim_files[i], dt),
            DPI,
            SCATTER_SIZE,
            SCATTER_ALPHA,
        )
        for i in range(n_frames)
    ]

    print(f"Rendering {n_frames} frames with {n_workers} workers (DT={dt})...")
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futs = [ex.submit(render_one_frame, t) for t in tasks]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="Rendering frames"):
            fut.result()

    # ffmpeg image2 demuxer requires contiguous numbering.
    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-i", str(png_dir / "frame_%06d.png"),
        # libx264 + yuv420p require even width/height; tight bbox can be odd.
        "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-crf", "23",
        out_mp4,
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except FileNotFoundError as e:
        raise RuntimeError("ffmpeg not found on PATH; required to stitch PNGs into MP4.") from e
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"ffmpeg failed:\n{e.stderr}") from e

    shutil.rmtree(cache_dir, ignore_errors=True)
    return out_mp4


In [ ]:
# Run animation pipeline (3D: x, y, v_x) with parallel rendering

init_file, anim_files = list_snapshot_files()
left_ids, right_ids = classify_beams(init_file)
keep_ids, left_ids, right_ids = subsample_ids(left_ids, right_ids)
raw_frames = load_animation_frames(anim_files)
xlim, ylim, vxlim = compute_axis_limits(raw_frames)
frames, rho_lim = prepare_phase_space_density(raw_frames, keep_ids)
del raw_frames  # free CSV data before render

saved_path = render_animation_parallel(
    frames, anim_files, xlim, ylim, vxlim, rho_lim
)
display(Video(saved_path, embed=True))
print(f"Saved animation to: {saved_path}")
